# B2 - Person B, GPU runs for the compact residual U-Net (denoise-study)

**Kaggle session settings:** Select the available **T4 x2** accelerator. B2 explicitly selects `cuda:0`; the second T4 is not used by the single-device training code. Internet **ON** only for the clone (or use an attached code dataset) | Inputs: `bsd400.zip` from Person A's `data/BSD400/`, and the saved `denoise-B-results` version containing the completed P checkpoints (required for inference).

B2 trains and validates on BSD400 only. The repository's pinned `artifacts/manifest.json` is the split source of truth; the bootstrap checks all 400 BSD400 file hashes. Do not attach Person A's full repo, legacy `bsd400_split.json`, old `manifest.csv`, BSD68, or old `runs/`/`outputs/` for B2. Person A's legacy files are not the current `denoise-shared` format.

**Current run:** train G seeds 0 and 1 with shared lambda 0.5, then infer P and G for both seeds. The B pilot metrics and A's lambda decision are recorded in `docs/lambda_pilot.md`; confirm the shared value with A before final reporting.
No debugging on the GPU quota: if something breaks, fix it on CPU (B1 / laptop), push, re-run.
Never run two GPU notebooks at once (same quota drains twice as fast). Turn the accelerator off when finished.

In [ ]:
# ---- CONFIG: edit this cell only ----
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle only offers T4 x2; use the first T4 only

NOTEBOOK_NAME = "B2_gpu_unet"
REPO_URL = "https://github.com/abhinavpareek655/denoise-study.git"  # repo URL; private repo -> use Kaggle Secrets, never paste a token here.
BRANCH = "main"
PINNED_COMMIT = "5b8d4cb"            # shared lambda 0.5 and training protocol

USE_SHARED_FOR_PIPELINE = False   # B2 uses the pinned repo manifest; only enable for a compatible current-format shared artifacts dataset
REQUIRE_GPU = True

# Requires BSD400 input and the saved denoise-B-results version with P checkpoints.
RUN = ["train_G_s0", "train_G_s1", "infer_P_s0", "infer_P_s1", "infer_G_s0", "infer_G_s1"]

SESSION_LIMIT_H = 9.0         # plan for <= 9 h (Section 24); the real Kaggle limit is up to 12 h
MAX_HOURS_PER_RUN = 8.5       # passed as --max-hours to full training runs (clean checkpoint + exit)

In [ ]:
# ---- Bootstrap: helpers, repo clone/pin, BSD400 lookup, env vars ----
import os, sys, json, time, shutil, hashlib, subprocess, datetime, importlib, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")          # the ONLY directory Kaggle keeps as notebook output
REPO = WORK / "denoise-study"
LOG_DIR = WORK / "notebook_logs"
LOG_DIR.mkdir(exist_ok=True)
assert Path("/kaggle").exists(), "This notebook is written for Kaggle (paths come from /kaggle)."


def sh(cmd, cwd=None, check=True, log=None):
    """Run a shell command, stream output live, optionally tee it to a log file. Returns exit code."""
    print(f"$ {cmd}", flush=True)
    t0 = time.time()
    run_dir = cwd or (REPO if REPO.exists() else None)
    p = subprocess.Popen(cmd, shell=True, cwd=run_dir, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lf = open(log, "a") if log else None
    for line in p.stdout:
        print(line, end="", flush=True)
        if lf:
            lf.write(line)
            lf.flush()
    rc = p.wait()
    if lf:
        lf.close()
    print(f"[exit {rc} after {time.time() - t0:.0f}s]", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc


def find_dir_containing(name, root="/kaggle/input", max_depth=3):
    """Breadth-first search under `root` for a directory that contains `name`."""
    queue = [(Path(root), 0)]
    while queue:
        d, depth = queue.pop(0)
        if not d.is_dir():
            continue
        if (d / name).exists():
            return d
        if depth < max_depth:
            queue.extend((c, depth + 1) for c in sorted(d.iterdir()) if c.is_dir())
    return None


# --- code: clone from GitHub (Internet ON) or copy from an attached code dataset (Internet OFF) ---
if not REPO.exists():
    if REPO_URL:
        assert "<" not in REPO_URL, "Edit REPO_URL in the config cell first."
        sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO}", cwd=WORK)
    else:
        src = find_dir_containing("PROJECT_CONTEXT.md")
        assert src is not None, "No REPO_URL and no attached code dataset containing PROJECT_CONTEXT.md"
        shutil.copytree(src, REPO)
if (REPO / ".git").exists() and REPO_URL:
    sh("git fetch --all --prune")
    sh(f"git checkout {PINNED_COMMIT}" if PINNED_COMMIT else f"git checkout {BRANCH} && git pull --ff-only")
COMMIT = subprocess.run("git rev-parse HEAD", shell=True, cwd=REPO, capture_output=True,
                        text=True).stdout.strip() or "no-git"
print("code commit:", COMMIT)

# --- data: B2 needs BSD400 only; BSD68 is not used for training or validation ---
DATA = find_dir_containing("BSD400")
if DATA is None:
    archives = sorted(Path("/kaggle/input").rglob("bsd400.zip"))
    assert archives, "Attach a Kaggle dataset containing BSD400/ or a file named bsd400.zip."
    EXTRACTED_DATA = WORK / "bsd-data"
    EXTRACTED_DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as zf:
        extraction_root = EXTRACTED_DATA.resolve()
        for member in zf.infolist():
            target = (EXTRACTED_DATA / member.filename).resolve()
            assert target.is_relative_to(extraction_root), f"unsafe path in BSD400 archive: {member.filename}"
        zf.extractall(EXTRACTED_DATA)
    DATA = find_dir_containing("BSD400", root=str(EXTRACTED_DATA))
assert DATA is not None and (DATA / "BSD400").is_dir(), "BSD400/ was not found in the attached input."
os.environ["DENOISE_DATA"] = str(DATA)

# Verify the BSD400 content against the pinned repo manifest before spending GPU time.
sys.path.insert(0, str(REPO))
from common.io import sha256_file
from common.manifest import load_manifest, select, validate_manifest

data_manifest = load_manifest(REPO / "artifacts" / "manifest.json")
validate_manifest(data_manifest)
train_records = select(data_manifest, "BSD400")
assert len(train_records) == 400, f"manifest has {len(train_records)} BSD400 records, expected 400"
bad_data = [r["relative_path"] for r in train_records
            if sha256_file(DATA / r["relative_path"]) != r["sha256"]]
assert not bad_data, f"BSD400 files do not match the pinned manifest: {bad_data[:5]}"
print(f"BSD400 verified: {len(train_records)} manifest-matched images from {DATA}")

# Optional read-only shared artifacts. Person A's legacy repo output folders are not compatible here.
SHARED_DIR = find_dir_containing("manifest.json")
if USE_SHARED_FOR_PIPELINE:
    assert SHARED_DIR is not None, "USE_SHARED_FOR_PIPELINE=True but no current-format shared artifacts dataset is attached"
    os.environ["DENOISE_SHARED"] = str(SHARED_DIR)
print("DENOISE_DATA  =", os.environ["DENOISE_DATA"])
print("DENOISE_SHARED=", os.environ.get("DENOISE_SHARED", "(unset: repo artifacts/ is used)"))
print("shared dataset found at:", SHARED_DIR)

os.chdir(REPO)
paths = importlib.import_module("common.paths")
print("\ncommon.paths resolves (CHECK these look right before running anything):")
for n in sorted(dir(paths)):
    v = getattr(paths, n)
    if not n.startswith("_") and isinstance(v, (str, Path, dict)):
        print(f"  {n} = {v}")
print("  functions:", [n for n in dir(paths) if not n.startswith("_") and callable(getattr(paths, n))])

In [ ]:
# ---- Environment capture (Section 24.3): saved to artifacts/logs/env_<timestamp>.json ----
import platform
import numpy, scipy, skimage, pandas


def _cap(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
    return r.stdout.strip()


env = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "notebook": NOTEBOOK_NAME,
    "code_commit": COMMIT,
    "python": platform.python_version(),
    "numpy": numpy.__version__, "scipy": scipy.__version__,
    "scikit_image": skimage.__version__, "pandas": pandas.__version__,
    "cpu_model": _cap("lscpu | grep 'Model name' | head -1 | cut -d: -f2"),
    "cpu_count": os.cpu_count(),
    "gpu": _cap("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") or "none",
    "data_dir": str(DATA), "shared_dir": str(SHARED_DIR),
}
try:
    import torch
    env.update(torch=torch.__version__, cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except Exception as e:  # torch missing is reported, not fatal, in the CPU notebook
    env["torch"] = f"unavailable: {e!r}"
(REPO / "artifacts" / "logs").mkdir(parents=True, exist_ok=True)
env_path = REPO / "artifacts" / "logs" / f"env_{datetime.datetime.now():%Y%m%d_%H%M%S}.json"
env_path.write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

# The contract wants common/versions.py output here; its API is not in the notes, so list what exists:
try:
    vers = importlib.import_module("common.versions")
    print("common.versions public names:", [n for n in dir(vers) if not n.startswith("_")])
except Exception as e:
    print("common.versions failed to import:", repr(e))


In [ ]:
import torch
GPU_COUNT = torch.cuda.device_count()
print("cuda available:", torch.cuda.is_available(), "| visible GPUs:", GPU_COUNT)
if REQUIRE_GPU:
    assert torch.cuda.is_available() and GPU_COUNT > 0, "No GPU: select Kaggle's T4 x2 accelerator."
    torch.cuda.set_device(0)
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print("GPU:", GPU_NAME, "| selected device: cuda:0; additional visible GPUs are not used")

## Restore checkpoints from a previous session
`/kaggle/working` starts **empty** in every new session. To resume (or to run inference on trained models), attach your
previous `denoise-B-results` version as an input; this cell copies `artifacts/checkpoints/<model_id>/` back so that
re-running the *same* `scripts.train` command resumes from `last.pt`. Attach **one** results version only.

In [ ]:
checkpoint_inputs = sorted(Path("/kaggle/input").rglob("last.pt"))
print("checkpoint inputs found:", [str(p) for p in checkpoint_inputs] if checkpoint_inputs else "none")
restored = []
for last in checkpoint_inputs:
    mid = last.parent.name
    dst = REPO / "artifacts" / "checkpoints" / mid
    if not dst.exists():
        shutil.copytree(last.parent, dst)
        restored.append(mid)
print("restored checkpoints:", restored if restored else "none (no new checkpoints copied)")
checkpoints = REPO / "artifacts" / "checkpoints"
for d in sorted(p for p in checkpoints.glob("*") if p.is_dir()) if checkpoints.exists() else []:
    print("  ", d.name, sorted(p.name for p in d.iterdir()))

required_p = ["unet_P_s0", "unet_P_s1"] if any(name.startswith("infer_P_") for name in RUN) else []
missing_p = [mid for mid in required_p
             if not (checkpoints / mid / "best.pt").is_file() or not (checkpoints / mid / "last.pt").is_file()]
if missing_p:
    attached = sorted(p.name for p in Path("/kaggle/input").iterdir())
    raise FileNotFoundError(
        f"Missing P checkpoints {missing_p}. Attach the saved denoise-B-results dataset version "
        f"containing artifacts/checkpoints/<model_id>/(best.pt,last.pt). Kaggle inputs found: {attached}")

## Job runner
This run uses the shared lambda 0.5 and Person A's completed-run settings: 20,000 final steps, batch size 64, and cosine learning-rate decay to 1e-6. It trains G for seeds 0 and 1, then runs inference for P and G for both seeds. Attach the previous `denoise-B-results` version so the completed P checkpoints are restored before inference.

Training is resumable: if either G run does not reach 20,000 steps, save a Kaggle version and select the same training preset in the next session. Confirm lambda 0.5 with Person A before final reporting.

In [ ]:
sh("python -m scripts.train --help | head -n 40", check=False)
sh("python -m scripts.infer --help | head -n 25", check=False)

from common import protocol as PR
SESSION_START = time.time()

PRESETS = {
    "tests": ["python -m pytest -q tests/test_torch_parts.py"],
    "bench": ["python -m scripts.train --arch unet --variant P --seed 0 --bench 200"],
    "pilot": [f"python -m scripts.train --arch unet --variant G --lam {lam} --pilot" for lam in PR.LAMBDA_GRID],
}
for v in PR.LOSS_VARIANTS:
    for s in PR.TRAIN_SEEDS:
        PRESETS[f"train_{v}_s{s}"] = [f"python -m scripts.train --arch unet --variant {v} --seed {s}"]
        PRESETS[f"infer_{v}_s{s}"] = [f"python -m scripts.infer --method unet --model-id unet_{v}_s{s}"]


def hours_left():
    return SESSION_LIMIT_H - (time.time() - SESSION_START) / 3600


def run_preset(name):
    if name.startswith("train_G"):
        pr = importlib.reload(importlib.import_module("common.protocol"))
        assert pr.LAMBDA_FROZEN is not None, (
            "LAMBDA_FROZEN is None: finish the pilot, set it ONCE in common/protocol.py (both networks), "
            "commit, push, then re-clone with the new commit.")
        print("LAMBDA_FROZEN =", pr.LAMBDA_FROZEN)
    if name.startswith("infer_"):
        model_id = f"unet_{name.removeprefix('infer_')}"
        checkpoint = REPO / "artifacts" / "checkpoints" / model_id / "best.pt"
        assert checkpoint.is_file(), (
            f"Missing {checkpoint}. Attach the denoise-B-results version with trained checkpoints, "
            "or complete the corresponding training preset first.")
    for cmd in PRESETS[name]:
        if "scripts.train" in cmd and "--bench" not in cmd and "--pilot" not in cmd:
            budget = min(MAX_HOURS_PER_RUN, hours_left() - 0.25)
            if budget < 0.5:
                print(f"Only {hours_left():.2f} h left in this session: not starting {name}.")
                return
            cmd += f" --max-hours {budget:.2f}"
        sh(cmd, log=LOG_DIR / f"{name}.log")

In [ ]:
for name in RUN:
    assert name in PRESETS, f"unknown preset {name}; valid: {sorted(PRESETS)}"
for name in RUN:
    print(f"\n########## {name}   (session time left: {hours_left():.2f} h) ##########")
    run_preset(name)

## Review what the run produced
Prints `meta.json`, the tail of `train_log.csv`, and saves a validation curve per model to `artifacts/figures/`.
Sanity rules (Section 14.6): validation PSNR must beat the noisy input at all three sigmas and should clearly beat the best
classical result on validation images; if not, suspect a bug (scale, residual sign, normalization, too few steps).

In [ ]:
import re
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = REPO / "artifacts" / "figures"
FIG.mkdir(parents=True, exist_ok=True)


def review_run(d):
    print(f"\n=== {d.name} ===  files: {sorted(p.name for p in d.iterdir())}")
    meta = d / "meta.json"
    hw = None
    if meta.exists():
        m = json.loads(meta.read_text())
        hw = m.get("hardware")
        print(json.dumps(m, indent=1)[:1500])
    else:
        print("meta.json not present yet")
    lg = d / "train_log.csv"
    if not lg.exists():
        return hw
    df = pd.read_csv(lg)
    print(df.tail(5).to_string(index=False))
    step = next((c for c in df.columns if "step" in c.lower()), None)
    vals = [c for c in df.columns if "val" in c.lower() and "psnr" in c.lower()]
    if step and vals:
        ax = df.plot(x=step, y=vals, figsize=(6, 3.5), marker="o", ms=3)
        ax.set_xlabel("training step")
        ax.set_ylabel("validation PSNR (dB)")
        ax.set_title(d.name)
        out = FIG / f"train_curve_{d.name}.png"
        ax.figure.savefig(out, dpi=200, bbox_inches="tight")
        plt.show()
        print("saved", out)
    return hw


ckdir = REPO / "artifacts" / "checkpoints"
hardware = {}
for d in sorted(ckdir.glob("unet_*")) if ckdir.exists() else []:
    hardware[d.name] = review_run(d)

for s in PR.TRAIN_SEEDS:
    a, b = hardware.get(f"unet_P_s{s}"), hardware.get(f"unet_G_s{s}")
    if a and b and a != b:
        print(f"WARNING seed {s}: P trained on '{a}' but G on '{b}' (Section 24.8: not bit-comparable)")

In [ ]:
def du_mb(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e6


for sub in ("checkpoints", "outputs", "logs", "figures"):
    p = REPO / "artifacts" / sub
    if p.exists():
        print(f"artifacts/{sub:11s} {du_mb(p):8.1f} MB")
print(f"/kaggle/working total: {du_mb(WORK):.0f} MB   (limit is about 20 GB)   session time used: {(time.time()-SESSION_START)/3600:.2f} h")

## After the run (not automatic)
1. **Pilot:** review `artifacts/tuning/lambda_pilot_unet.csv` and the three pilot checkpoint logs using B's 20 validation images. Person A's repo selected 0.5, but A's validation split shares no images with B's manifest; agree on one shared lambda with A before freezing it in `common/protocol.py`.
2. **G training:** after that protocol change is committed and pushed, pin the new commit and set `RUN` to `train_G_s0` and `train_G_s1`. P and G for each seed use the same 20,000-step budget.
3. **Training outputs:** each completed model should have `best.pt`, `last.pt`, `meta.json`, and `train_log.csv` under `artifacts/checkpoints/unet_<variant>_s<seed>/`. Resume an incomplete run by selecting the same preset. Save a Kaggle version between sessions.
4. **Inference:** after the checkpoints are complete, select matching `infer_*` presets. Expect denoised PNGs under `artifacts/outputs/<model_id>/sigma{15,25,50}/` and runtime CSVs under `artifacts/results/`.
5. Publish checkpoints, outputs, and logs as the next `denoise-B-results` dataset version; record its version and fill `docs/model_provenance.md` from the saved `meta.json` files.